In [ ]:
# 1 - Instalar librerías
!pip install flask flask-ngrok scikit-learn joblib numpy

In [ ]:
# 2- Entrenar y guardar el modelo
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

# Cargar datos
X, y = load_iris(return_X_y=True)

# Dividir y entrenar el modelo
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Guardar modelo
joblib.dump(model, 'modelo_iris.pkl')

print("¡Modelo entrenado y guardado como 'modelo_iris.pkl'!")

In [ ]:
# 3 - Crear y ejecutar la API
from flask import Flask, request, jsonify
import joblib
import numpy as np
import threading
import time

# Inicializar app
app = Flask(__name__)

# Cargar el modelo guardado
model = joblib.load('modelo_iris.pkl')

# Definir el endpoint de predicción
@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    features = np.array(data['features']).reshape(1, -1)
    prediction = model.predict(features)
    return jsonify({'prediction': int(prediction[0])})

# Función para ejecutar la app de Flask
def run_flask_app():
    app.run(port=8000)

# Ejecutar la app de Flask en un hilo separado
if __name__ == '__main__':
    # Crea un hilo para ejecutar la app de Flask
    thread = threading.Thread(target=run_flask_app)
    thread.daemon = True  # Permite que el hilo principal salga aunque el hilo de Flask se esté ejecutando
    thread.start()
    # Dale al servidor un momento para iniciar
    time.sleep(1)
    print("La app de Flask se está ejecutando en un hilo separado en el puerto 8000.")

In [ ]:
# 4 - Probar la API
import requests
import time

# Usa el puerto donde se ejecuta la app de Flask
ngrok_url = 'http://127.0.0.1:8000'

# Los datos que quieres enviar para la predicción
test_data = {
    "features": [5.1, 3.5, 1.4, 0.2]
}

# Dale al servidor un momento para iniciar
time.sleep(2)

# Hacemos la solicitud POST al endpoint /predict de nuestra API
response = requests.post(f'{ngrok_url}/predict', json=test_data)

# Imprimimos la respuesta
print(response.json())